In [3]:
!pip install -q ultralytics kagglehub

import os
import random
import shutil
from pathlib import Path
from ultralytics import YOLO
import kagglehub

print("✅ Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Setup complete


In [ ]:
path = kagglehub.dataset_download("dakshivashishtha/exdark-occluded")

print("Dataset path:", path)

100%|██████████| 4.85G/4.85G [00:49<00:00, 106MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1


In [ ]:
base = path

DATA_PATH = os.path.join(base, "occ50")

img_dir = os.path.join(DATA_PATH, "images")
lbl_dir = os.path.join(DATA_PATH, "labels")

print("Images:", img_dir)
print("Labels:", lbl_dir)

Images: /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1/occ50/images
Labels: /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1/occ50/labels


In [ ]:
def split_dataset(img_dir, lbl_dir, output):

    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)
    output = Path(output)

    images = list(img_dir.glob("*"))
    random.shuffle(images)

    n = len(images)
    train_split = int(0.7 * n)
    val_split   = int(0.9 * n)

    splits = {
        "train": images[:train_split],
        "val":   images[train_split:val_split],
        "test":  images[val_split:]
    }

    for split, imgs in splits.items():
        (output/"images"/split).mkdir(parents=True, exist_ok=True)
        (output/"labels"/split).mkdir(parents=True, exist_ok=True)

        for img_path in imgs:
            label_path = lbl_dir / (img_path.stem + ".txt")

            if not label_path.exists():
                continue

            shutil.copy(img_path, output/"images"/split/img_path.name)
            shutil.copy(label_path, output/"labels"/split/label_path.name)

    print("✅ Split complete")


split_dataset(img_dir, lbl_dir, "dataset_split")

✅ Split complete


In [ ]:
import shutil
from google.colab import files

zip_name = "dataset_split"

shutil.make_archive(zip_name, 'zip', "dataset_split")

files.download(f"{zip_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

def count_split_data(root="dataset_split"):
    for split in ["train", "val", "test"]:
        img_path = os.path.join(root, "images", split)
        lbl_path = os.path.join(root, "labels", split)

        n_imgs = len(os.listdir(img_path))
        n_lbls = len(os.listdir(lbl_path))

        print(f"{split.upper()}:")
        print(f"  Images: {n_imgs}")
        print(f"  Labels: {n_lbls}")
        print("-" * 25)

count_split_data()

TRAIN:
  Images: 5152
  Labels: 5152
-------------------------
VAL:
  Images: 1472
  Labels: 1472
-------------------------
TEST:
  Images: 737
  Labels: 737
-------------------------


In [ ]:
yaml_content = """
path: dataset_split
train: images/train
val: images/val
test: images/test

nc: 12
names: [Bicycle, Boat, Bottle, Bus, Car, Cat, Chair, Cup, Dog, Motorbike, People, Table]
"""

with open("dataset_baseline.yaml", "w") as f:
    f.write(yaml_content)

print("✅ YAML created")

✅ YAML created


In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data="dataset_baseline.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    name="baseline_model"
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_baseline.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_model, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bb1a4abbfb0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,  

In [ ]:
metrics = model.val()
print(metrics)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,988 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1476.3±571.7 MB/s, size: 150.7 KB)
val: Scanning /content/dataset_split/labels/val.cache... 2650 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 2650/2650 855.0Mit/s 0.0s
val: /content/dataset_split/images/val/2015_05337.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [      1.075]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 166/166 5.2it/s 31.8s
                   all       2649       8654      0.888      0.818      0.904      0.656
               Bicycle        263        393      0.912      0.818      0.916      0.681
                  Boat        256        513      0.942      0.908      0.968      0.673
                Bottle        262        534      0.873      0.772     

In [ ]:
model.predict("dataset_split/images/test", save=True)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1402 /content/dataset_split/images/test/2015_00002.png: 448x640 4 Bicycles, 2 Cars, 44.8ms
image 2/1402 /content/dataset_split/images/test/2015_00008.jpg: 640x448 1 Bicycle, 1 People, 48.8ms
image 3/1402 /content/dataset_split/images/test/2015_00010.jpg: 448x640 1 Bicycle, 1 People, 6.6ms
image 4/1402 /content/dataset_split/images/test/2015_00024.JPG: 480x640 3 Bicycles, 39.4ms
image 5/1402 /content/dataset_split/images/test/2015_00027.jpg: 480x6

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Bicycle', 1: 'Boat', 2: 'Bottle', 3: 'Bus', 4: 'Car', 5: 'Cat', 6: 'Chair', 7: 'Cup', 8: 'Dog', 9: 'Motorbike', 10: 'People', 11: 'Table'}
 obb: None
 orig_img: array([[[ 10,   5,   6],
         [  9,   5,   4],
         [  7,   6,   2],
         ...,
         [  7,   5,   5],
         [  7,   5,   4],
         [  7,   5,   4]],
 
        [[  9,   4,   5],
         [  7,   5,   4],
         [  5,   4,   0],
         ...,
         [  8,   7,   3],
         [  8,   7,   3],
         [  8,   7,   3]],
 
        [[  6,   4,   4],
         [  6,   4,   3],
         [  6,   5,   1],
         ...,
         [  7,   6,   2],
         [  7,   6,   2],
         [  7,   6,   2]],
 
        ...,
 
        [[  4,  56, 123],
         [  4,  42, 138],
         [  3,  39, 133],
         ...,
         [  3,  36, 115],
         [  4,  37, 117],
         

In [ ]:
import shutil
from google.colab import files

zip_name = "baseline_results"

shutil.make_archive(zip_name, 'zip', "runs/detect/baseline_model")

files.download(f"{zip_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
!unzip baseline_results.zip

Archive:  baseline_results.zip
   creating: weights/
  inflating: labels.jpg              
  inflating: BoxPR_curve.png         
  inflating: confusion_matrix.png    
  inflating: val_batch2_pred.jpg     
  inflating: BoxF1_curve.png         
  inflating: confusion_matrix_normalized.png  
  inflating: val_batch0_labels.jpg   
  inflating: train_batch4192.jpg     
  inflating: results.png             
  inflating: train_batch0.jpg        
  inflating: val_batch0_pred.jpg     
  inflating: val_batch2_labels.jpg   
  inflating: results.csv             
  inflating: BoxP_curve.png          
  inflating: args.yaml               
  inflating: train_batch4191.jpg     
  inflating: train_batch2.jpg        
  inflating: train_batch1.jpg        
  inflating: BoxR_curve.png          
  inflating: train_batch4190.jpg     
  inflating: val_batch1_labels.jpg   
  inflating: val_batch1_pred.jpg     
  inflating: weights/last.pt         
  inflating: weights/best.pt         


In [16]:
!unzip occ_50.zip

Streaming output truncated to the last 5000 lines.
  inflating: labels/train/2015_04742.txt  
  inflating: labels/train/2015_00546.txt  
  inflating: labels/train/2015_01909.txt  
  inflating: labels/train/2015_04470.txt  
  inflating: labels/train/2015_02243.txt  
  inflating: labels/train/2015_02799.txt  
  inflating: labels/train/2015_03036.txt  
  inflating: labels/train/2015_04068.txt  
  inflating: labels/train/2015_03563.txt  
  inflating: labels/train/2015_03183.txt  
  inflating: labels/train/2015_05454.txt  
  inflating: labels/train/2015_03250.txt  
  inflating: labels/train/2015_00919.txt  
  inflating: labels/train/2015_01728.txt  
  inflating: labels/train/2015_02333.txt  
  inflating: labels/train/2015_00677.txt  
  inflating: labels/train/2015_02220.txt  
  inflating: labels/train/2015_06267.txt  
  inflating: labels/train/2015_04593.txt  
  inflating: labels/train/2015_05572.txt  
  inflating: labels/train/2015_00891.txt  
  inflating: labels/train/2015_02515.txt  
  i

In [17]:
import shutil
import os

os.makedirs("occ50", exist_ok=True)

shutil.move("images", "occ50/images")
shutil.move("labels", "occ50/labels")

'occ50/labels'

In [12]:
from ultralytics import YOLO

model_base = YOLO("weights/best.pt")

In [18]:
data_yaml = """
path: /content/occ50

train: images/train
val: images/val
test: images/test

nc: 12
names: ['Bicycle','Boat','Bottle','Bus','Car','Cat','Chair','Cup','Dog','Motorbike','People','Table']
"""

with open("occ50.yaml", "w") as f:
    f.write(data_yaml)

print("✅ occ50.yaml ready")

✅ occ50.yaml ready


In [19]:
metrics_base_occ50 = model_base.val(data="occ50.yaml", split="test")

results_base = {}

results_base["occ50_test"] = {
    "mAP50": metrics_base_occ50.box.map50,
    "mAP50-95": metrics_base_occ50.box.map,
    "Precision": metrics_base_occ50.box.mp,
    "Recall": metrics_base_occ50.box.mr
}

print("✅ OCC50 done")

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2042.1±922.1 MB/s, size: 190.9 KB)
val: Scanning /content/occ50/labels/test... 737 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 737/737 1.9Kit/s 0.4s
val: New cache created: /content/occ50/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 4.7it/s 10.0s
                   all        737       2437      0.877      0.843      0.909      0.669
               Bicycle         78        112      0.844      0.893       0.94      0.714
                  Boat         73        147      0.962      0.959      0.971      0.683
                Bottle         62        157       0.87      0.808      0.879       0.59
                   Bus         60         73      0.868          1      0.992      0.832
                   Car        120        251      0.899      0.916      0.961   

In [21]:
import kagglehub

path = kagglehub.dataset_download("dakshivashishtha/exdark-occluded")

print("Dataset path:", path)

100%|██████████| 4.85G/4.85G [02:07<00:00, 40.8MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1


In [24]:
base = path   # already from kagglehub

def create_yaml(name, path):
    yaml = f"""
path: {path}

train: images
val: images
test: images

nc: 12
names: [Bicycle, Boat, Bottle, Bus, Car, Cat, Chair, Cup, Dog, Motorbike, People, Table]
"""
    with open(f"{name}.yaml", "w") as f:
        f.write(yaml)

# create missing YAMLs
create_yaml("clean", base + "/clean")
create_yaml("occ25", base + "/occ25")
create_yaml("occ75", base + "/occ75")

print("✅ clean, occ25, occ75 YAML created")

✅ clean, occ25, occ75 YAML created


In [25]:
for d in ["clean", "occ25", "occ75"]:
    m = model_base.val(data=f"{d}.yaml")

    results_base[d] = {
        "mAP50": m.box.map50,
        "mAP50-95": m.box.map,
        "Precision": m.box.mp,
        "Recall": m.box.mr
    }

    print(f"\n📊 {d.upper()} DONE")
    print(results_base[d])

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 33.8±23.7 MB/s, size: 85.3 KB)
val: Scanning /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1/clean/labels... 7361 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 7361/7361 561.9it/s 13.1s
val: /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1/clean/images/2015_05337.jpeg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [      1.075]
val: New cache created: /root/.cache/kagglehub/datasets/dakshivashishtha/exdark-occluded/versions/1/clean/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 460/460 6.1it/s 1:15
                   all       7360      23702      0.648       0.29      0.367      0.221
               Bicycle        750       1118      0.839      0.358      0.499      0.296
                  Boat

In [26]:
import pandas as pd

df_base = pd.DataFrame(results_base).T

# add F1 score
df_base["F1"] = 2 * (df_base["Precision"] * df_base["Recall"]) / (df_base["Precision"] + df_base["Recall"] + 1e-6)

df_base = df_base.round(4)

print("🔥 FINAL BASELINE RESULTS")
print(df_base)

# save file
df_base.to_csv("baseline_results.csv")

🔥 FINAL BASELINE RESULTS
             mAP50  mAP50-95  Precision  Recall      F1
occ50_test  0.9086    0.6685     0.8774  0.8435  0.8601
clean       0.3671    0.2213     0.6479  0.2901  0.4008
occ25       0.1157    0.0516     0.1719  0.2475  0.2029
occ75       0.2990    0.0854     0.4202  0.3539  0.3842


In [27]:
import shutil

shutil.make_archive("runs_backup", 'zip', "runs")

print("✅ runs zipped")

✅ runs zipped


In [28]:
from google.colab import files

files.download("baseline_results.csv")
files.download("runs_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>